# Case Attribute Distributions from Real Log

In [1]:
import pm4py
import pandas as pd
import numpy as np
from scipy import stats

In [2]:
log_path = "../data/BPI Challenge 2017.xes.gz"
log = pm4py.read_xes(log_path)
df = pm4py.convert_to_dataframe(log)
print(f"Events: {len(df):,}, Cases: {df['case:concept:name'].nunique():,}")

/Users/zeynepcetin/bppso-groupwork-1/.venv/lib/python3.13/site-packages/pm4py/utils.py:1000: UserWarning: Install the optional requirement `rustxes` to import/export files faster.
  warnings.warn(
/Users/zeynepcetin/bppso-groupwork-1/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 31509/31509 [00:16<00:00, 1909.37it/s]


Events: 1,202,267, Cases: 31,509


In [3]:
# Case-level: one row per case
case_df = df.groupby("case:concept:name").first().reset_index()

# Loan goal distribution

In [4]:
if "case:LoanGoal" in case_df.columns:
    loan_counts = case_df["case:LoanGoal"].fillna("Unknown").value_counts(normalize=True)
    print("LOAN_GOAL_DIST = {")
    for goal, pct in loan_counts.items():
        print(f'    "{goal}": {pct:.4f},')
    print("}")
else:
    print("case:LoanGoal column not found!")

LOAN_GOAL_DIST = {
    "Car": 0.2960,
    "Home improvement": 0.2434,
    "Existing loan takeover": 0.1778,
    "Other, see explanation": 0.0947,
    "Unknown": 0.0751,
    "Not speficied": 0.0338,
    "Remaining debt home": 0.0267,
    "Extra spending limit": 0.0198,
    "Caravan / Camper": 0.0117,
    "Motorcycle": 0.0087,
    "Boat": 0.0064,
    "Tax payments": 0.0048,
    "Business goal": 0.0010,
    "Debt restructuring": 0.0001,
}


# Application type dist

In [5]:
if "case:ApplicationType" in case_df.columns:
    app_counts = case_df["case:ApplicationType"].fillna("Unknown").value_counts(normalize=True)
    print("APP_TYPE_DIST = {")
    for app, pct in app_counts.items():
        print(f'    "{app}": {pct:.4f},')
    print("}")
else:
    print("case:ApplicationType column not found!")

APP_TYPE_DIST = {
    "New credit": 0.8924,
    "Limit raise": 0.1076,
}


# Requested amount (Lognormal fit)

In [6]:
if "case:RequestedAmount" in case_df.columns:
    amounts = pd.to_numeric(case_df["case:RequestedAmount"], errors="coerce").dropna()
    amounts = amounts[amounts > 0]

    print(f"RequestedAmount stats:")
    print(f"  count: {len(amounts)}")
    print(f"  mean:  {amounts.mean():.0f}")
    print(f"  median:{amounts.median():.0f}")
    print(f"  min:   {amounts.min():.0f}")
    print(f"  max:   {amounts.max():.0f}")

    log_amounts = np.log(amounts)
    mu = log_amounts.mean()
    sigma = log_amounts.std()

    print(f"\nLognormal fit:")
    print(f"  AMOUNT_LOG_MEAN = {mu:.4f}")
    print(f"  AMOUNT_LOG_STD  = {sigma:.4f}")
else:
    print("case:RequestedAmount column not found!")

RequestedAmount stats:
  count: 28429
  mean:  17993
  median:15000
  min:   600
  max:   450000

Lognormal fit:
  AMOUNT_LOG_MEAN = 9.5387
  AMOUNT_LOG_STD  = 0.7092


# Credit score dist

In [7]:
if "CreditScore" in df.columns:
    # CreditScore is per-offer, get max per case
    cs_per_case = df.groupby("case:concept:name")["CreditScore"].max()
    total_cases = case_df["case:concept:name"].nunique()
    has_score = cs_per_case.dropna()

    pct_with_score = len(has_score) / total_cases
    print(f"Cases with CreditScore: {len(has_score)} / {total_cases} ({pct_with_score:.2%})")

    scores = has_score.values.astype(float)
    print(f"\nCreditScore stats (for cases that have one):")
    print(f"  mean:  {scores.mean():.1f}")
    print(f"  std:   {scores.std():.1f}")
    print(f"  min:   {scores.min():.0f}")
    print(f"  max:   {scores.max():.0f}")

    # Bin distribution
    bins = pd.cut(scores, bins=[0, 400, 600, 800, float("inf")],
                  labels=["poor", "fair", "good", "excellent"])
    bin_dist = pd.Series(bins).value_counts(normalize=True)
    print(f"\nBin distribution:")
    for b, pct in bin_dist.items():
        print(f"  {b}: {pct:.4f}")

    print(f"\n# For spawner:")
    print(f"# {pct_with_score:.2%} of cases have a credit score")
    print(f"# Gaussian fit: mean={scores.mean():.1f}, std={scores.std():.1f}")
else:
    print("CreditScore column not found!")

Cases with CreditScore: 31509 / 31509 (100.00%)

CreditScore stats (for cases that have one):
  mean:  434.7
  std:   454.0
  min:   0
  max:   1145

Bin distribution:
  excellent: 0.8289
  good: 0.1699
  fair: 0.0012
  poor: 0.0000

# For spawner:
# 100.00% of cases have a credit score
# Gaussian fit: mean=434.7, std=454.0


# Amount Category dist

In [8]:
if "case:RequestedAmount" in case_df.columns:
    amounts = pd.to_numeric(case_df["case:RequestedAmount"], errors="coerce").fillna(0)
    cats = pd.cut(
        amounts,
        bins=[0, 5000, 10000, 20000, 50000, float("inf")],
        labels=["very_low", "low", "medium", "high", "very_high"]
    )
    print("Amount category distribution:")
    print(cats.value_counts(normalize=True).sort_index())

Amount category distribution:
case:RequestedAmount
very_low     0.150515
low          0.249077
medium       0.317035
high         0.253685
very_high    0.029688
Name: proportion, dtype: float64
